# 03 -- All-sky background (floating dark matter)

Notebooks 01 and 02 restrict the background to **up-going** muons, because the
standard EarthShine signal is dark matter captured in the Earth's core, which
sends muons up through the detector from a single direction.

"Floating" dark matter models relax that: the annihilation region is not
pinned to the Earth's centre, so the signal can arrive from any zenith angle.
This notebook drops the `hemisphere="up"` restriction and asks what the
background looks like across the whole sky.

**The short version, established below:** the neutrino-induced background is
roughly flat across the sky and the up/down asymmetry is mild. But that is not
the relevant comparison, because going *down* you are no longer competing with
neutrino-induced muons at all -- you are competing with ordinary cosmic-ray
muons, which outnumber them by **seven orders of magnitude** at a 100 m
overburden. Any all-sky search is really a cosmic-ray-veto problem, not a
neutrino-background problem.

In [ ]:
# --- make `nubkg` importable no matter where this notebook is opened from ----
import sys, pathlib

def _bootstrap():
    here = pathlib.Path.cwd()
    for cand in [here, *here.parents][:4]:
        if (cand / "nubkg" / "__init__.py").exists():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    return None

_root = _bootstrap()
print("package root:", _root or "not found next to the notebook")

import numpy as np
import matplotlib.pyplot as plt
import nubkg as nb
from nubkg import plots as P

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.titlesize": 10, "figure.autolayout": True})

E_NU_RANGE = (10.0, 1.0e6)
# ---- input tables, all in nubkg/data/ -- change the names HERE, nowhere else ----
FLUX_FILE = "atmospheric_numu_digitised.csv"   # E, E^2*Phi, err
XSEC_FILE = "xsec_cc_digitised.csv"            # E, sigma_nu, sigma_nubar
nb.set_default_xsec(XSEC_FILE)                 # every calculation now uses it
print(f"flux table: {FLUX_FILE}   cross-section table: {nb.get_default_xsec()}")
LIVETIME = nb.SEC_PER_YEAR
#cms = nb.Cylinder(radius=7.5, half_length=10.5, depth=100.0, name="CMS-like")
cms = nb.Cylinder(radius=1.12, half_length=2.7, depth=100.0, name="CMS-like (inner tracker)")
rock = nb.STANDARD_ROCK

# all-sky flux: the zenith shape is now grafted using the FULL-SKY average,
# not the up-going one, so the normalisation is preserved over 4pi
flux = nb.TabulatedFlux.from_csv(
    nb.data_path(FLUX_FILE),
    label="digitised atmospheric, all-sky", flux_unit="E2Phi",
    zenith_shape_from=nb.ChirkinAtmospheric(), shape_hemisphere="all")
print(f"{cms.name}: R={cms.radius} m, L={2*cms.half_length} m, "
      f"depth={cms.depth} m ({cms.depth*100*rock.density/1e5:.3f} km.w.e.)")

## 1. Neutrino-induced muons across the whole sky

`hemisphere="all"` integrates cos(zenith) from -1 to +1. Two things change
relative to the up-going case:

* **down-going neutrinos are not absorbed** by the Earth (no chord to cross),
  which slightly *raises* the flux at high energy;
* **down-going muons have limited rock available** -- only the overburden,
  `depth/cos(theta)` -- which lowers the rate, and increasingly so as the
  threshold rises and the required muon range grows.

In [ ]:
print(f"{'E_thr [GeV]':>12} {'up':>10} {'down':>10} {'all-sky':>10} {'down/up':>9}")
allsky = {}
for t in (10., 100., 1000., 5000.):
    kw = dict(flux=flux, e_mu_min=t, e_nu_range=E_NU_RANGE,
              livetime_s=LIVETIME, uncertainty=False)
    u = nb.background_muons(cms, hemisphere="up",   **kw)["n_muons"]
    d = nb.background_muons(cms, hemisphere="down", **kw)["n_muons"]
    a = nb.background_muons(cms, hemisphere="all",  **kw)["n_muons"]
    allsky[t] = a
    print(f"{t:12.0f} {u:10.2f} {d:10.2f} {a:10.2f} {d/u:9.3f}")

### Zenith distribution

This is the plot that matters for a directional search: where in the sky the
background sits, per unit solid angle.

In [ ]:
def zenith_profile(e_thr, n_cos=120):
    cz = np.linspace(-1, 1, n_cos)
    cz = np.where(np.abs(cz) < 1e-6, 1e-6, cz)
    xm = cms.max_rock_path_gcm2(cz, rock)
    phi = nb.integrated_muon_flux(e_thr, cz, flux=flux, rock=rock,
                                  max_path_gcm2=xm, e_nu_range=E_NU_RANGE)
    area = cms.mean_projected_area_m2(cz) * 1e4
    return cz, phi, phi * area * 2 * np.pi * LIVETIME

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for t, c in zip((10., 100., 1000.), ("C0", "C1", "C2")):
    cz, phi, dr = zenith_profile(t)
    axes[0].plot(cz, phi, color=c, label=f"$E_\\mu$ > {t:g} GeV")
    axes[1].plot(cz, dr, color=c, label=f"$E_\\mu$ > {t:g} GeV")
for ax, yl in zip(axes, [r"$\Phi_\mu$ [cm$^{-2}$s$^{-1}$sr$^{-1}$]",
                         r"$dN/d\cos\theta$ per year"]):
    ax.set_yscale("log"); ax.set_xlabel(r"$\cos\theta_{\rm zenith}$")
    ax.set_ylabel(yl); ax.grid(alpha=.25); ax.legend(fontsize=7)
    ax.axvline(0, color="k", lw=.8, ls=":")
    ax.text(-0.5, ax.get_ylim()[1]*0.3, "up-going", ha="center", fontsize=8)
    ax.text(0.5, ax.get_ylim()[1]*0.3, "down-going", ha="center", fontsize=8)
axes[0].set_title("neutrino-induced muon flux vs zenith", fontsize=10)
axes[1].set_title("rate through the detector vs zenith", fontsize=10)
plt.show()

The up/down asymmetry is mild -- less than a factor of two at a 10 GeV
threshold -- and grows with threshold as the overburden cap bites harder.
Note the flux peaks near the **horizon** (cos θ ≈ 0), which is the atmospheric
horizon enhancement: near-horizontal parent mesons travel further in thin
atmosphere before interacting, so more of them decay to neutrinos.

## 2. The thing that actually decides the answer: cosmic-ray muons

Going down, the competition is not other neutrinos. It is cosmic-ray muons
that punched through the overburden. `nubkg.cosmic_muons` estimates these by
taking the Gaisser surface muon spectrum and requiring survival through
`depth/cos(theta)` of rock, using **the same `EnergyLoss` object** as the
neutrino side, so the comparison is apples to apples.

This is an order-of-magnitude tool -- see the module docstring for the list of
things it does not do. For the point being made here, that is ample.

In [ ]:
print("cosmic-ray vs neutrino-induced muons through the detector, per year:")
print(f"{'E_thr [GeV]':>12} {'CR (down)':>14} {'nu (all-sky)':>14} {'CR / nu':>12}")
for t in (10., 100., 1000., 5000.):
    cr = nb.cosmic_muon_rate(cms, t, rock=rock)["rate_per_year"]
    print(f"{t:12.0f} {cr:14.3e} {allsky[t]:14.2f} {cr/allsky[t]:12.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
cz, phi, dr = zenith_profile(10.)
ax.plot(cz, np.maximum(dr, 1e-30), color="crimson", lw=1.8,
        label=r"$\nu$-induced, $E_\mu$ > 10 GeV")
# max_sec_theta must be large here, or the cap flattens the curve near the
# horizon and hides the crossover; 1e4 lets the slant depth grow physically
czc = np.geomspace(1e-4, 1, 400)
crf = nb.muon_flux_at_depth(10., czc, cms.depth, rock.density, max_sec_theta=1e4)
crr = crf * cms.mean_projected_area_m2(czc) * 1e4 * 2 * np.pi * LIVETIME
ax.plot(czc, np.maximum(crr, 1e-30), color="navy", lw=1.8,
        label=r"cosmic-ray muons, $E_\mu$ > 10 GeV")
ax.plot([0.0192], [np.interp(0.0192, cz, dr)], "o", ms=7, mfc="none",
        mec="k", mew=1.5)
ax.annotate("they cross 1.1$\degree$ above\nthe horizon", xy=(0.0192, 3e2),
            xytext=(0.30, 1e1), fontsize=7.5,
            arrowprops=dict(arrowstyle="->", lw=.9))
ax.set_yscale("log"); ax.set_ylim(1e-2, 1e12)
ax.set_xlabel(r"$\cos\theta_{\rm zenith}$")
ax.set_ylabel(r"$dN/d\cos\theta$ per year")
ax.axvline(0, color="k", lw=.8, ls=":")
ax.axvspan(-1, 0, color="green", alpha=.06)
ax.text(-0.5, 1e10, "up-going:\nno cosmic-ray muons\n(Earth absorbs them)",
        ha="center", fontsize=8, color="darkgreen")
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=.25)
ax.set_title("why the up-going hemisphere is the signal region", fontsize=10)
plt.show()

That gap is the entire reason the standard analysis looks up. Cosmic-ray muons
are *identically zero* from below -- the Earth absorbs them -- so up-going the
neutrino-induced flux is the irreducible background. Down-going it is
irrelevant, buried under something 10⁷ times larger.

## 3. How close to the horizon must you be?

If the floating-DM signal arrives near-horizontally, the slant depth grows as
sec(θ) and cosmic-ray muons die fast. Where do the two cross?

In [ ]:
czc = np.geomspace(1e-4, 1.0, 300)
xm = cms.max_rock_path_gcm2(czc, rock)
for t in (10., 100., 1000.):
    nuf = nb.integrated_muon_flux(t, czc, flux=flux, rock=rock,
                                  max_path_gcm2=xm, e_nu_range=E_NU_RANGE)
    crf = nb.muon_flux_at_depth(t, czc, cms.depth, rock.density,
                                max_sec_theta=1e4)
    r = crf / nuf
    czx = np.interp(0.0, np.log(np.maximum(r, 1e-300)), czc)
    print(f"E_thr={t:6.0f} GeV: CR = nu at cos(zen) = {czx:.5f}  "
          f"= {90-np.degrees(np.arccos(czx)):.2f} deg above the horizon")
print()
print("Solid angle of a band that many degrees above the horizon, as a")
print("fraction of the down-going hemisphere:")
for deg in (1.1, 5., 10.):
    print(f"   within {deg:4.1f} deg of horizontal: "
          f"{np.sin(np.radians(deg)):.3f}")

So the usable down-going sky is a sliver within ~1 degree of horizontal --
about 2% of the down-going hemisphere -- and *exactly* where the crossing
falls depends on site topography, which a flat-overburden model cannot
supply. CMS sits under a largely flat plain near Cessy, but the Jura is close
enough that a real analysis needs a topographic overburden map.

**Practical conclusion for floating-DM models:** treat the whole down-going
hemisphere as inaccessible without a cosmic-ray veto. The numbers in notebook
02 are the right ones for any signal region at or above the horizon. If a
model puts signal below the horizon, the analysis becomes a veto-efficiency
study, and the neutrino background is no longer the limiting factor.

## 4. All-sky rates, for the record

If the veto problem is solved (or the model only needs the up-going plus
horizontal sky), these are the neutrino-induced numbers with full
uncertainties.

In [ ]:
e_mu_min = 100

res = nb.background_muons(cms, e_mu_min=e_mu_min, e_nu_range=E_NU_RANGE,
                          livetime_s=LIVETIME, flux=flux, hemisphere="all")
print(f"N(all-sky, E_mu > {e_mu_min} GeV, 1 yr) = {res['n_muons']:.1f} "
      f"+- {res['n_muons_err']:.1f}")
for k, v in sorted(res["error_budget"].items(), key=lambda kv: -kv[1]):
    print(f"   {k:15s} {v:7.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
P.plot_arriving_spectrum(res, ax=axes[0])
_, scan = P.plot_threshold_scan(np.geomspace(10, 5e3, 12), cms, ax=axes[1],
                                livetime_s=LIVETIME, flux=flux,
                                e_nu_range=E_NU_RANGE, hemisphere="all")
axes[1].set_title("all-sky threshold scan", fontsize=10)
plt.show()

## 5. Caveats specific to this notebook

1. **The cosmic-ray estimate is order-of-magnitude.** Gaisser's surface flux is
   quoted as valid for E > 100/cos θ GeV, and a 100 m overburden only selects
   surface energies of ~70 GeV, so it is being used just below its floor. The
   cross-check against Mei & Hime (`pytest -m literature`) agrees only to
   within an order of magnitude, and the two bracket rather than confirm each
   other -- Gaisser is below its energy floor, Mei & Hime below its depth
   floor. Neither is reliable at 0.265 km.w.e. For a real number use
   MUSIC/MUSUN or CORSIKA with a site overburden map.
2. **Flat overburden.** `depth/cos(theta)` diverges at the horizon while real
   rock paths do not. Everything within a few degrees of horizontal is
   topography-dependent.
3. **No cosmic-ray veto modelling.** CMS has one, and it is good; none of it is
   represented here.
4. The neutrino side carries all the caveats from notebooks 01 and 02.